<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/XML%20for%20Contracts/07-lab-xml-contracts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 7 Lab — XML for Contracts: Schema Validation (XSD) and XPath Extraction

This lab applies the **parse → validate → extract** chain to a simplified e-invoicing scenario. You read an XSD contract by filling in element contract cards, predict validation outcomes before running the validator, compare XSD and DTD validation, extract fields with XPath, validate JSON with JSON Schema, and complete a cross-model comparison.

Use the formal vocabulary (well-formed, valid, element, attribute, compositor, cardinality, contract card, XPath, axis, node test, predicate, namespace, validator rejects) throughout your work.

**Time budget:** Part 1 ~20 min | Part 2 ~25 min | Part 3 ~10 min | Part 4 ~25 min | Part 5 ~15 min | Part 6 ~15 min | **Total ~110 min**

**Tools required:** Google Colab with `cellspell` (`%%xpath` magic for xmllint, Python for JSON Schema)

---

## Setup

Run the following cells in a new Google Colab notebook. All XML, XSD, and DTD files are created via `%%writefile` cells — you do not need to upload anything.

**Cell 1 — Install cellspell:**

In [9]:
# Topic 7 — XML for Contracts: Schema Validation and XPath Extraction
!apt-get update -qq && apt-get install -y -qq libxml2-utils > /dev/null 2>&1
!pip install "cellspell[xpath] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
%load_ext cellspell.xpath

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The cellspell.xpath extension is already loaded. To reload it, use:
  %reload_ext cellspell.xpath



**Cell 2 — Verify xmllint:**

In [10]:
!xmllint --version 2>&1 | head -1
# Expected: xmllint using libxml version ...

xmllint: using libxml version 20913



**Cell 3 — Download XML contract files from course repository:**


In [11]:
# ── Download XML/XSD/DTD files from course repository ──────
import urllib.request, os

BASE_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/e-invoicing/"
FILES = ["invoice_schema.xsd", "invoice_valid.xml", "invoice_invalid.xml",
         "invoice.dtd", "invoice_namespaced.xml"]

for fname in FILES:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE_URL + fname, fname)
        print(f"Downloaded {fname} ({os.path.getsize(fname):,} bytes)")

print("All contract files ready.")

All contract files ready.



---

## Part 1 — Read the Contract (XSD Comprehension)

Before validating any XML, read the XSD. This part builds the skill from Requirement 2: systematic contract reading via element contract cards.

### Q1. Contract card — `Invoice` (root element)

Read `invoice_schema.xsd` and fill in the contract card for the root `Invoice` element:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children (in order) | |
| Restrictions | |
| Attributes | |

**Hint:** The root element uses an anonymous complex type (defined inline). List all children declared in the `xs:sequence`, including their cardinalities.

### Q2. Contract card — `InvoiceLine`

Fill in the contract card for `InvoiceLine`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children (in order) | |
| Restrictions | |
| Attributes | |

### Q3. Contract card — `Quantity`

Fill in the contract card for `Quantity`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes | |

**Key question:** What values does `xs:positiveInteger` accept? Would `0` pass? Would `-3` pass? Would `5.5` pass?

### Q4. Contract card — `PaymentMethodCode`

Fill in the contract card for `PaymentMethodCode`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes | |

**Key question:** List the four allowed values. Would `"bank_transfer"` pass? Would `"30 "` (with trailing space) pass?

### Q5. Contract card — `LineAmount`

Fill in the contract card for `LineAmount`. This element has both text content (a decimal) and an attribute (`currencyID`).

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes (name, type, required?) | |

**Key question:** `AmountType` uses `xs:simpleContent` with `xs:extension`. What does this mean structurally? How does it differ from a plain `xs:string` element?

---

## Part 2 — Validate with XSD (Predict, Then Verify)

### Q6. Well-formedness check — both documents

Run the well-formedness check on both documents. **Predict** the result before running:

In [12]:
%xpath invoice_valid.xml

✓ Valid


In [13]:
%xpath invoice_invalid.xml

✓ Valid



**Record:** Are both well-formed? Why would an invalid document still be well-formed?

### Q7. Predict validation failures (the four-check model)

**Before running the validator**, read `invoice_invalid.xml` alongside the XSD. Using the four-check model from the narrative (name, compositor, cardinality, content), identify each error.

Fill in the table:

| Error # | What's wrong | XSD rule violated | Which check? (name / compositor / cardinality / content) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

**Important:** Complete this table entirely from reading the XSD and XML — do not run the validator yet.

### Q8. Validate and compare

Now validate the invalid document:

In [14]:
%xpath --xsd invoice_schema.xsd invoice_invalid.xml

✗ Validation failed:
invoice_invalid.xml:14: element MonetaryTotal: Schemas validity error : Element 'MonetaryTotal': This element is not expected. Expected is ( InvoiceLines ).
invoice_invalid.xml fails to validate



**Record the actual errors.** Compare to your predictions in Q7:

- Which errors did you correctly predict?
- Did the validator report errors in the same order as your predictions?
- Did any error cascade (one error causing the validator to report additional errors)?

### Q9. Validate the valid document

Confirm that the valid invoice passes:

In [15]:
%xpath --xsd invoice_schema.xsd invoice_valid.xml

✓ Valid
invoice_valid.xml validates



**Expected:** `✓ Valid`. If this fails, check the XSD and XML files from the Setup cells.

---

## Part 3 — DTD Comparison (What DTD Cannot Express)

### Q10. Validate with DTD — invalid document

Validate the invalid invoice against the DTD:

In [16]:
%xpath --dtd invoice.dtd invoice_invalid.xml

✗ Validation failed:
invoice_invalid.xml:2: element Invoice: validity error : Element Invoice content does not follow the DTD, expecting (InvoiceNumber , IssueDate , Supplier , Customer , InvoiceLines , MonetaryTotal , PaymentMeans* , Note?), got (InvoiceNumber IssueDate Supplier Customer MonetaryTotal InvoiceLines PaymentMeans Note Note )
Document invoice_invalid.xml does not validate against invoice.dtd



**Before running, predict:** Which of the four errors from Q7 will the DTD catch? Which will it miss? Why?

**Record the actual DTD output.** Fill in:

| Error # | XSD catches? | DTD catches? | Why DTD misses it (if applicable) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

### Q11. DTD limitation analysis

In 2–3 sentences, explain: Why do standards bodies (UBL, XBRL, HL7) use XSD instead of DTD? Reference the specific errors from Q10 that DTD missed to support your answer.

---

## Part 4 — XPath Extraction (Predict, Then Verify)

All XPath queries in this part run against `invoice_valid.xml` (the validated document).

### Q12. Basic path — supplier name

**Predict** the result, then run:

In [17]:
%%xpath invoice_valid.xml
/Invoice/Supplier/Name/text()

Kopi Tech Pte Ltd




Parse this expression using the axis–node-test–predicate model:
- Axis steps: ?
- Node tests: ?
- Predicates: ?

### Q13. Descendant axis — all line amounts

**Predict** how many results and what values:

In [18]:
%%xpath invoice_valid.xml
//InvoiceLine/LineAmount/text()

12000.00
250.00
3000.00




**Question:** Why does `//` (descendant) work here instead of the full path `/Invoice/InvoiceLines/InvoiceLine/LineAmount`? When would you prefer the full path?

### Q14. Predicate filter — items with quantity > 3

**Predict** the result count and values:

In [19]:
%%xpath invoice_valid.xml
/Invoice/InvoiceLines/InvoiceLine[Quantity > 3]/ItemName/text()

SSL Certificate
Technical Support (Hours)




**Verify:** Does the result match the prediction from the narrative checkpoint (Section 3.7)?

### Q15. Attribute extraction — currency codes

**Predict** the result:

In [20]:
%%xpath invoice_valid.xml
/Invoice/MonetaryTotal/PayableAmount/@currencyID

 currencyID="SGD"




**Question:** How does `@currencyID` differ from a child element? Could the XSD have modelled currency as a child element `<CurrencyCode>SGD</CurrencyCode>` instead of an attribute? What would change in the XPath?

### Q16. Namespace trap — unqualified path on namespaced document

Run the following XPath on the **namespaced** invoice:

In [21]:
%%xpath invoice_namespaced.xml
/Invoice/Supplier/Name/text()

XPath error: XPath set is empty



**Predict:** Will this return "Kopi Tech Pte Ltd" or empty?

Now run with the namespace bound:

In [22]:
%%xpath invoice_namespaced.xml
/inv:Invoice/inv:Supplier/inv:Name/text()

XPath error: XPath error : Undefined namespace prefix
XPath evaluation failure



(Note: `cellspell` automatically binds prefixes declared in the document.)

**Record:** What changed? In 1–2 sentences, explain why unqualified paths fail on namespaced documents.

---

## Part 5 — JSON Schema Validation (The Modern Equivalent)

This part connects to the narrative's Section 4.7 and resolves Failure 3.

### Q17. Validate with JSON Schema

Run the following cell to validate the malformed payment notification from Failure 3 against the JSON Schema:

In [23]:
import json
from jsonschema import validate, ValidationError, Draft202012Validator

schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "required": ["amount", "currency", "paymentDate", "merchantId"],
    "properties": {
        "amount":      {"type": "number", "exclusiveMinimum": 0},
        "currency":    {"type": "string", "enum": ["SGD", "USD", "EUR", "GBP"]},
        "paymentDate": {"type": "string", "format": "date"},
        "merchantId":  {"type": "string", "pattern": "^M-\\d{4}$"}
    },
    "additionalProperties": False
}

# The malformed payload from Failure 3
payload = {
    "amount": "150.00",
    "currency": "sgd",
    "payment_date": "2025-02-13",
    "merchant_id": "M-2001"
}

validator = Draft202012Validator(schema)
errors = list(validator.iter_errors(payload))
for e in errors:
    print(f"  Path: {list(e.path) or '(root)'}  |  Error: {e.message}")
print(f"\nTotal errors: {len(errors)}")

  Path: (root)  |  Error: 'paymentDate' is a required property
  Path: (root)  |  Error: 'merchantId' is a required property
  Path: ['amount']  |  Error: '150.00' is not of type 'number'
  Path: ['currency']  |  Error: 'sgd' is not one of ['SGD', 'USD', 'EUR', 'GBP']
  Path: (root)  |  Error: Additional properties are not allowed ('merchant_id', 'payment_date' were unexpected)

Total errors: 5



**Before running, predict:** How many errors will the validator report? Map each expected error to a specific JSON Schema rule (type, required, enum, additionalProperties).

**After running:** Compare actual errors to your predictions. Which errors match the XSD four-check model from Part 2?

### Q18. Fix and re-validate

Create a corrected payload that passes validation:

In [24]:
corrected = {
    "amount": 150.00,
    "currency": "SGD",
    "paymentDate": "2025-02-13",
    "merchantId": "M-2001"
}

errors = list(validator.iter_errors(corrected))
if not errors:
    print("✓ Valid")
else:
    for e in errors:
        print(f"  Error: {e.message}")

✓ Valid



**Record:** What did you change? Map each fix to the JSON Schema rule it satisfies.

---

## Part 6 — Closing Bridge (Cross-Model Comparison)

### Q19. Complete the comparison table

Using your experience from this lab and Chapters 5–6, fill in the table:

| Dimension | Relational (Ch 5) | JSON / MongoDB (Ch 6) | XML / XSD (Ch 7) | JSON Schema (Ch 7) |
|---|---|---|---|---|
| Schema posture | | | | |
| Enforcement spectrum position | | | | |
| When validation runs | | | | |
| What it catches | | | | |
| Primary context | | | | |
| Who controls the schema? | | | | |

### Q20. Applied scenario — two workloads

A Singapore fintech company has two data flows:

1. **Internal API:** Mobile app sends payment events to the backend (JSON, same team controls both ends, schema evolves weekly).
2. **Regulatory reporting:** Backend submits monthly transaction summaries to MAS (Monetary Authority of Singapore) via an XML portal with a published XSD.

For each flow, recommend:
- (a) Which validation approach? (engine rejects / validator rejects with XSD / validator rejects with JSON Schema / no validation)
- (b) Justify your choice using the trust boundary distinction from Section 4.5.
- (c) What is the cost of choosing the wrong approach? (Reference a specific failure from the narrative.)

---

## Submission Checklist

Before submitting, verify you have:

- [ ] Five completed contract cards (Q1–Q5) with all fields filled
- [ ] Prediction table for invalid invoice errors (Q7) completed **before** running validator
- [ ] XSD validation output recorded with comparison to predictions (Q8)
- [ ] DTD limitation table (Q10) with explanations for what DTD misses
- [ ] DTD limitation analysis paragraph (Q11)
- [ ] Five XPath results with axis–node-test–predicate parsing (Q12–Q16)
- [ ] Namespace trap demonstrated with explanation (Q16)
- [ ] JSON Schema validation with error predictions and comparison (Q17)
- [ ] Corrected JSON payload passing validation (Q18)
- [ ] Cross-model comparison table (Q19)
- [ ] Applied scenario analysis with justified recommendations (Q20)


---

## Solutions

Complete your own work before reviewing the solutions below. Each explanation references specific XSD rules and the four-check model.

---

### Part 1 Solutions

<details>
<summary><strong>Q1 Solution — Invoice (root element)</strong></summary>

| Field | Answer |
|---|---|
| Name | `Invoice` |
| Parent | (root — no parent) |
| Compositor | `xs:sequence` |
| Cardinality | exactly 1 (root element) |
| Type | anonymous complex type (defined inline) |
| Children (in order) | `InvoiceNumber` (1), `IssueDate` (1), `Supplier` (1), `Customer` (1), `InvoiceLines` (1), `MonetaryTotal` (1), `PaymentMeans` (0..*), `Note` (0..1) |
| Restrictions | Children must appear in sequence order |
| Attributes | none |

</details>

---

<details>
<summary><strong>Q2 Solution — InvoiceLine</strong></summary>

| Field | Answer |
|---|---|
| Name | `InvoiceLine` |
| Parent | `InvoiceLines` |
| Compositor | `xs:sequence` (via `InvoiceLineType`) |
| Cardinality | 1..* (`minOccurs="1" maxOccurs="unbounded"`) |
| Type | `InvoiceLineType` (named complex type) |
| Children (in order) | `ItemName` (xs:string), `Quantity` (xs:positiveInteger), `LineAmount` (AmountType) |
| Restrictions | Children must appear in sequence order |
| Attributes | none (but `LineAmount` child has `currencyID` attribute) |

</details>

---

<details>
<summary><strong>Q3 Solution — Quantity</strong></summary>

| Field | Answer |
|---|---|
| Name | `Quantity` |
| Parent | `InvoiceLine` |
| Compositor | n/a (simple type) |
| Cardinality | exactly 1 (default) |
| Type | `xs:positiveInteger` |
| Children | none (leaf element) |
| Restrictions | Must be a positive integer (1, 2, 3, ...) |
| Attributes | none |

`xs:positiveInteger` accepts 1, 2, 3, ... (integers > 0). `0` would fail (not positive). `-3` would fail (negative). `5.5` would fail (not an integer).

</details>

---

<details>
<summary><strong>Q4 Solution — PaymentMethodCode</strong></summary>

| Field | Answer |
|---|---|
| Name | `PaymentMethodCode` |
| Parent | `PaymentMeans` |
| Compositor | n/a (simple type) |
| Cardinality | exactly 1 |
| Type | `PaymentMethodCodeType` (named simple type) |
| Children | none (leaf element) |
| Restrictions | `xs:enumeration` — allowed values: `"30"`, `"42"`, `"48"`, `"1"` |
| Attributes | none |

`"bank_transfer"` would fail — not in the enumeration. `"30 "` (with trailing space) would also fail — XSD string enumerations are exact match (no whitespace normalisation on enumerated values).

</details>

---

<details>
<summary><strong>Q5 Solution — LineAmount</strong></summary>

| Field | Answer |
|---|---|
| Name | `LineAmount` |
| Parent | `InvoiceLine` |
| Compositor | n/a (simple content with extension) |
| Cardinality | exactly 1 |
| Type | `AmountType` (named complex type) |
| Children | none (text content only — `xs:decimal`) |
| Restrictions | Text content must be a valid `xs:decimal` |
| Attributes | `currencyID` — type `CurrencyCodeType`, required (`use="required"`) |

`xs:simpleContent` with `xs:extension` means the element has **text content** (a decimal value) **plus** an attribute. It differs from a plain `xs:string` because it carries structured metadata (the currency) alongside the value. This is the XML equivalent of a typed field with a unit annotation.

</details>

---

### Part 2 Solutions

<details>
<summary><strong>Q6 Solution — Well-formedness</strong></summary>

Both documents are well-formed. Well-formedness checks XML syntax (matching tags, proper nesting, valid character encoding) — it does **not** check schema rules. A document can be well-formed XML but invalid against its schema. The invalid invoice has correct XML syntax; its errors are all schema-level violations.

</details>

---

<details>
<summary><strong>Q7 Solution — Four errors in invoice_invalid.xml</strong></summary>

| Error # | What's wrong | XSD rule violated | Which check? |
|---|---|---|---|
| 1 | `MonetaryTotal` appears before `InvoiceLines` | `xs:sequence` requires children in declared order | compositor |
| 2 | `Quantity` is `-3` | `xs:positiveInteger` requires values > 0 | content |
| 3 | `PaymentMethodCode` is `"bank_transfer"` | `PaymentMethodCodeType` enumeration: only `"30"`, `"42"`, `"48"`, `"1"` | content |
| 4 | Two `Note` elements | `maxOccurs="1"` on `Note` | cardinality |

</details>

---

<details>
<summary><strong>Q8 Solution — Validate and compare</strong></summary>

The XSD validator reports all four errors. The first error (compositor — wrong element order) typically causes a cascade: the validator reports "Element 'MonetaryTotal' is not expected. Expected is 'InvoiceLines'" and may report subsequent elements as unexpected too. The actual number of reported messages may exceed four due to cascading, but there are exactly four root-cause errors.

</details>

---

<details>
<summary><strong>Q9 Solution — Valid document</strong></summary>

`✓ Valid` — the valid invoice conforms to all XSD rules.

</details>

---

### Part 3 Solutions

<details>
<summary><strong>Q10 Solution — DTD validation</strong></summary>

| Error # | XSD catches? | DTD catches? | Why DTD misses it |
|---|---|---|---|
| 1 | Yes | Yes | DTD enforces element order via its content model |
| 2 | Yes | No | DTD has no data types — `Quantity` is `#PCDATA` (any text). DTD cannot express "positive integer." |
| 3 | Yes | No | DTD has no enumeration on element content — `PaymentMethodCode` is `#PCDATA`. DTD enumerations only work on attributes. |
| 4 | Yes | Yes | DTD content model `Note?` enforces at most one occurrence |

DTD catches 2 of the 4 errors (compositor and cardinality). It misses the 2 content-type errors.

</details>

---

<details>
<summary><strong>Q11 Solution — DTD limitations</strong></summary>

Standards bodies use XSD because it supports **data types** (integer, decimal, date) and **value-space restrictions** (enumerations on element content, range constraints, pattern facets). DTD treats all element content as untyped text (`#PCDATA`), so it cannot enforce that a quantity is a positive integer or that a payment code comes from a closed list. For machine-to-machine contracts where data integrity is critical, these type-level guarantees are essential.

</details>

---

### Part 4 Solutions

<details>
<summary><strong>Q12 Solution — Supplier name</strong></summary>

Result: `Kopi Tech Pte Ltd`

The path `/Invoice/Supplier/Name/text()` navigates: root → Supplier → Name → text content. The `text()` node test extracts the string value.

</details>

---

<details>
<summary><strong>Q13 Solution — All line amounts</strong></summary>

Result: `12000.00`, `250.00`, `3000.00`

`//InvoiceLine/LineAmount/text()` uses the descendant axis (`//`) to find all `InvoiceLine` elements regardless of depth. This works because `InvoiceLine` only appears inside `InvoiceLines`. The full path `/Invoice/InvoiceLines/InvoiceLine/LineAmount/text()` produces the same result but is more explicit. Prefer the full path when the document structure might have element name collisions at different levels.

</details>

---

<details>
<summary><strong>Q14 Solution — Predicate filter</strong></summary>

Result: `SSL Certificate`, `Technical Support (Hours)`

The predicate `[Quantity > 3]` filters InvoiceLine elements where the Quantity child has a value greater than 3. Only 2 of the 3 invoice lines match (the Network Switch has Quantity 1).

</details>

---

<details>
<summary><strong>Q15 Solution — Attribute extraction</strong></summary>

Result: `currencyID="SGD"`

`@currencyID` extracts the attribute value. An attribute differs from a child element in that it is metadata attached to the element's opening tag, not a separate node in the content tree. The XSD could have modelled currency as a child element `<CurrencyCode>SGD</CurrencyCode>`, but then the XPath would be `/Invoice/MonetaryTotal/PayableAmount/CurrencyCode/text()` instead of `.../@currencyID`. The attribute approach is more concise for single-value metadata.

</details>

---

<details>
<summary><strong>Q16 Solution — Namespace trap</strong></summary>

**(a)** The unqualified path `/Invoice/Supplier/Name/text()` returns empty on the namespaced document. This is because the elements are in the `urn:example:invoice` namespace. The unqualified name `Invoice` refers to the **no-namespace** name, which doesn't match `inv:Invoice`.

**(b)** With namespace-qualified paths (`/inv:Invoice/inv:Supplier/inv:Name/text()`), the query returns `Kopi Tech Pte Ltd`. Every element name must include the prefix that is bound to the correct namespace URI.

Unqualified paths fail on namespaced documents because XML namespaces change the **identity** of elements. `Invoice` and `inv:Invoice` are different names from XPath's perspective — the prefix binds to a URI, and that URI is part of the element's qualified name.

</details>

---

### Part 5 Solutions

<details>
<summary><strong>Q17 Solution — JSON Schema validation</strong></summary>

The validator reports 5 errors:

1. `'paymentDate' is a required property` — the field is missing entirely (required check)
2. `'merchantId' is a required property` — the field is missing entirely (required check)
3. `'150.00' is not of type 'number'` — the amount is a string `"150.00"` instead of a number `150.00` (type check)
4. `'sgd' is not one of ['SGD', 'USD', 'EUR', 'GBP']` — lowercase `"sgd"` doesn't match the enum (enum/content check)
5. Additional property `'extra_field'` is not allowed (additionalProperties check)

These map to the XSD four-check model: required ≈ cardinality, type ≈ content, enum ≈ content, additionalProperties ≈ name.

</details>

---

<details>
<summary><strong>Q18 Solution — Fix and re-validate</strong></summary>

Fixes applied:
- `amount`: changed from string `"150.00"` to number `150.00` (type rule)
- `currency`: changed from `"sgd"` to `"SGD"` (enum rule — case-sensitive)
- `paymentDate`: added `"2025-02-13"` (required rule)
- `merchantId`: added `"M-2001"` (required rule)
- `extra_field`: removed (additionalProperties rule)

Result: `✓ Valid`

</details>

---

### Part 6 Solutions

<details>
<summary><strong>Q19 Solution — Comparison table</strong></summary>

| Dimension | Relational (Ch 5) | JSON / MongoDB (Ch 6) | XML / XSD (Ch 7) | JSON Schema (Ch 7) |
|---|---|---|---|---|
| Schema posture | Schema-on-write | Schema-on-read (optional validation) | Schema-on-write (if XSD applied) | Schema-on-write (if validator applied) |
| Enforcement spectrum position | Strict (DDL + constraints) | Flexible (validation optional) | Strict (XSD) or Loose (DTD) | Moderate (application-level) |
| When validation runs | On every INSERT/UPDATE | On insert if validator configured | Before processing (parsing stage) | At application boundary |
| What it catches | Type, referential integrity, CHECK, UNIQUE | Type, required fields, enum, pattern | Type, structure, cardinality, content | Type, required, enum, pattern, additionalProperties |
| Primary context | OLTP databases | Document stores, APIs | B2B interchange, regulatory filing | REST APIs, microservices |
| Who controls the schema? | DBA / application team | Application team (optional) | Standards body or trading partner | API provider |

</details>

---

<details>
<summary><strong>Q20 Solution — Applied scenario</strong></summary>

**(a)** Government e-invoicing mandate: **XML + XSD**. Regulatory compliance requires a strict, externally-defined contract that both sender and receiver agree on. XSD enforces element order, data types, enumerations, and cardinality — all critical for machine-to-machine processing where malformed data causes rejected filings. The schema is published by the standards body (e.g., PEPPOL BIS), not controlled by either trading partner.

**(b)** Internal mobile app API: **JSON Schema**. JSON is the native format for mobile/web clients (no XML parsing overhead). JSON Schema validates payloads at the API boundary, catching type errors and missing fields before they reach business logic. The schema is controlled by the API team and can evolve independently. `additionalProperties: false` prevents unexpected fields, and the schema doubles as API documentation.

</details>
